<a href="https://colab.research.google.com/github/Y4ri3/Agrimeteo/blob/notebook-analyse/notebook_analyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Dossier de stockage des fichiers : Google Drive sur Colab, dossier local "donnees" sinon
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DOSSIER_DONNEES = "/content/drive/MyDrive/agrimeteo"
except ImportError:
    DOSSIER_DONNEES = "donnees"

os.makedirs(DOSSIER_DONNEES, exist_ok=True)
FICHIER_BRUT = f"{DOSSIER_DONNEES}/agri_meteo_brut_2015_2025.csv"
print("Dossier de données :", DOSSIER_DONNEES)

Mounted at /content/drive
Dossier de données : /content/drive/MyDrive/agrimeteo


Configuration du stockage. Tous les fichiers produits par le notebook (fichier brut, jeu de données propre, indicateurs, fixture, rapport qualité, profils YAML) sont enregistrés dans le dossier agrimeteo de Google Drive. Ils survivent ainsi à la fin de la session Colab, et l'équipe dev peut les récupérer au même endroit. En dehors de Colab, le notebook utilise un dossier local nommé donnees.

In [ ]:
import requests
import pandas as pd
import unicodedata
import time
from datetime import date
import os

# L'extraction n'est lancée que si le fichier brut n'existe pas déjà sur Drive
if not os.path.exists(FICHIER_BRUT):

    # Localites

    localites = [
        {"localite": "Dakar", "region": "Dakar", "zone": "Niayes"},
        {"localite": "Mboro", "region": "Thiès", "zone": "Niayes"},
        {"localite": "Saint-Louis", "region": "Saint-Louis", "zone": "Vallée du fleuve"},
        {"localite": "Matam", "region": "Matam", "zone": "Vallée du fleuve"},
        {"localite": "Linguère", "region": "Louga", "zone": "Ferlo"},
        {"localite": "Ranérou", "region": "Matam", "zone": "Ferlo"},
        {"localite": "Diourbel", "region": "Diourbel", "zone": "Bassin arachidier"},
        {"localite": "Kaolack", "region": "Kaolack", "zone": "Bassin arachidier"},
        {"localite": "Tambacounda", "region": "Tambacounda", "zone": "Sénégal oriental"},
        {"localite": "Kédougou", "region": "Kédougou", "zone": "Sénégal oriental"},
        {"localite": "Ziguinchor", "region": "Ziguinchor", "zone": "Casamance"},
        {"localite": "Kolda", "region": "Kolda", "zone": "Casamance"}
    ]


    # Parametres des API

    URL_GEOCODING = "https://geocoding-api.open-meteo.com/v1/search"
    URL_METEO = "https://archive-api.open-meteo.com/v1/archive"

    params_meteo = {
        "start_date": "2015-01-01",
        "end_date": "2025-12-31",

        "daily": [
            "precipitation_sum",
            "temperature_2m_min",
            "temperature_2m_max",
            "temperature_2m_mean",
            "relative_humidity_2m_mean",
            "et0_fao_evapotranspiration"
        ],

        "timezone": "Africa/Dakar"
    }


    # Fonction de normalisation des noms

    def normaliser(nom):
        nom = unicodedata.normalize("NFKD", nom)
        nom = "".join(
            caractere for caractere in nom
            if not unicodedata.combining(caractere)
        )
        return nom.lower().strip()


    # Recuperation automatique des coordonnees

    def obtenir_coordonnees(ville):

        params = {
            "name": ville["localite"],
            "count": 10,
            "language": "fr",
            "format": "json",
            "countryCode": "SN"
        }

        response = requests.get(
            URL_GEOCODING,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        donnees = response.json()

        candidats = donnees.get("results", [])

        # Garder uniquement les résultats situés au Sénégal
        candidats = [
            c for c in candidats
            if c.get("country_code") == "SN"
        ]

        # Chercher une correspondance exacte du nom
        correspondances = [
            c for c in candidats
            if normaliser(c["name"]) == normaliser(ville["localite"])
        ]

        if not correspondances:
            raise ValueError(
                f"Aucune correspondance exacte trouvée pour {ville['localite']}"
            )

        # Si plusieurs résultats existent, privilégier la région attendue
        correspondances_region = [
            c for c in correspondances
            if c.get("admin1")
            and normaliser(c["admin1"]) == normaliser(ville["region"])
        ]

        if correspondances_region:
            choix = correspondances_region[0]

        elif len(correspondances) == 1:
            choix = correspondances[0]

        else:
            raise ValueError(
                f"Plusieurs résultats possibles pour {ville['localite']}. "
                "Vérification manuelle nécessaire."
            )

        return {
            "latitude": choix["latitude"],
            "longitude": choix["longitude"],
            "nom_api": choix["name"],
            "region_api": choix.get("admin1")
        }


    # Extraction meteorologique par annee

    resultats = []
    annees_echouees = []

    for ville in localites:

        print(f"\nExtraction : {ville['localite']}...")

        try:
            # Récupération des coordonnées
            coordonnees = obtenir_coordonnees(ville)

            latitude = coordonnees["latitude"]
            longitude = coordonnees["longitude"]

            print(f"Coordonnées : {latitude}, {longitude}")

            donnees_localite = []

            # Extraction année par année
            for annee in range(2015, 2026):

                debut = f"{annee}-01-01"
                fin = "2025-12-31" if annee == 2025 else f"{annee}-12-31"

                print(f"  Année {annee}...", end=" ")

                succes = False

                # Jusqu'à 4 tentatives en cas d'erreur
                for tentative in range(1, 5):

                    try:
                        params = params_meteo.copy()

                        params["latitude"] = latitude
                        params["longitude"] = longitude
                        params["start_date"] = debut
                        params["end_date"] = fin

                        response = requests.get(
                            URL_METEO,
                            params=params,
                            timeout=300
                        )

                        response.raise_for_status()

                        # Vérifier que la réponse n'est pas vide
                        if not response.text.strip():
                            raise ValueError("Réponse vide de l'API")

                        data = response.json()

                        if "daily" not in data:
                            raise ValueError("Données journalières absentes")

                        df = pd.DataFrame(data["daily"])

                        if df.empty:
                            raise ValueError("Aucune donnée reçue")

                        donnees_localite.append(df)

                        print(f"{len(df)} jours récupérés.")
                        succes = True
                        break

                    except (requests.exceptions.RequestException, ValueError) as e:

                        print(f"\n    Tentative {tentative}/4 échouée : {e}")

                        if tentative < 4:
                            time.sleep(5 * tentative)

                if not succes:
                    print(f"    Échec définitif pour {annee}.")
                    annees_echouees.append({
                        "localite": ville["localite"],
                        "annee": annee
                    })

                time.sleep(2)

            # Fusion des années pour cette localité
            if donnees_localite:

                df = pd.concat(donnees_localite, ignore_index=True)

                df.rename(columns={
                    "time": "date",
                    "precipitation_sum": "precipitation_mm",
                    "temperature_2m_min": "tmin_c",
                    "temperature_2m_max": "tmax_c",
                    "temperature_2m_mean": "tmean_c",
                    "relative_humidity_2m_mean": "rh_mean_pct",
                    "et0_fao_evapotranspiration": "et0_mm"
                }, inplace=True)

                df["localite"] = ville["localite"]
                df["region"] = ville["region"]
                df["zone_agroecologique"] = ville["zone"]
                df["latitude"] = latitude
                df["longitude"] = longitude

                df["source"] = "Open-Meteo Historical Weather"
                df["date_extraction"] = date.today().isoformat()

                resultats.append(df)

                print(
                    f"Total pour {ville['localite']} : "
                    f"{len(df)} jours récupérés."
                )

        except Exception as e:
            print(f"Erreur générale pour {ville['localite']} : {e}")


    # Fusion et exportation

    if resultats:

        dataset = pd.concat(resultats, ignore_index=True)

        dataset["date"] = pd.to_datetime(dataset["date"])

        colonnes = [
            "date",
            "localite",
            "region",
            "zone_agroecologique",
            "latitude",
            "longitude",
            "precipitation_mm",
            "tmin_c",
            "tmax_c",
            "tmean_c",
            "rh_mean_pct",
            "et0_mm",
            "source",
            "date_extraction"
        ]

        dataset = dataset[colonnes]

        dataset.to_csv(
            FICHIER_BRUT,
            index=False,
            encoding="utf-8-sig"
        )

        print("EXTRACTION TERMINEE")

        print("Dimensions :", dataset.shape)
        print("Localités récupérées :", dataset["localite"].nunique())

        print("\nNombre de jours par localité :")
        print(dataset.groupby("localite")["date"].count())

        if annees_echouees:
            print("\nAnnées non récupérées :")
            print(pd.DataFrame(annees_echouees))

    else:
        print("Aucune donnée récupérée.")

else:
    print(f"Fichier brut déjà présent, extraction ignorée : {FICHIER_BRUT}")



Extraction : Dakar...
Coordonnées : 14.6937, -17.44406
  Année 2015... 365 jours récupérés.
  Année 2016... 366 jours récupérés.
  Année 2017... 365 jours récupérés.
  Année 2018... 365 jours récupérés.
  Année 2019... 365 jours récupérés.
  Année 2020... 366 jours récupérés.
  Année 2021... 365 jours récupérés.
  Année 2022... 365 jours récupérés.
  Année 2023... 365 jours récupérés.
  Année 2024... 366 jours récupérés.
  Année 2025... 365 jours récupérés.
Total pour Dakar : 4018 jours récupérés.

Extraction : Mboro...
Coordonnées : 15.14051, -16.88682
  Année 2015... 365 jours récupérés.
  Année 2016... 366 jours récupérés.
  Année 2017... 365 jours récupérés.
  Année 2018... 365 jours récupérés.
  Année 2019... 365 jours récupérés.
  Année 2020... 366 jours récupérés.
  Année 2021... 365 jours récupérés.
  Année 2022... 365 jours récupérés.
  Année 2023... 365 jours récupérés.
  Année 2024... 366 jours récupérés.
  Année 2025... 365 jours récupérés.
Total pour Mboro : 4018 jours ré

Cette cellule fait l'extraction complète depuis l'API Open-Meteo (archive). Les coordonnées de chaque localité sont retrouvées automatiquement par géocodage, puis les données sont téléchargées année par année avec jusqu'à quatre tentatives en cas d'erreur, ce qui évite de perdre une localité entière si une requête échoue. Le résultat est un fichier brut de 12 localités sur 2015-2025.

In [ ]:
import pandas as pd

dataset = pd.read_csv(FICHIER_BRUT)

On relit le fichier brut depuis Drive. Tant que ce fichier existe, la cellule d'extraction ne relance pas l'API : on évite ainsi de refaire l'extraction à chaque session Colab et de dépasser les quotas d'Open-Meteo. Pour forcer une nouvelle extraction, il suffit de supprimer le fichier brut dans le dossier Drive.

In [ ]:
dataset.head(15)

,date,localite,region,zone_agroecologique,latitude,longitude,precipitation_mm,tmin_c,tmax_c,tmean_c,rh_mean_pct,et0_mm,source,date_extraction
0,2015-01-01,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,19.2,27.1,22.5,48,5.27,Open-Meteo Historical Weather,2026-11-11
1,2015-01-02,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,18.8,26.8,22.3,35,6.07,Open-Meteo Historical Weather,2026-11-11
2,2015-01-03,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,19.2,28.7,23.6,33,6.29,Open-Meteo Historical Weather,2026-11-11
3,2015-01-04,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,21.0,27.0,23.7,34,5.90,Open-Meteo Historical Weather,2026-11-11
4,2015-01-05,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,21.1,26.4,23.1,44,6.45,Open-Meteo Historical Weather,2026-11-11
5,2015-01-06,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,20.1,25.8,22.5,43,6.30,Open-Meteo Historical Weather,2026-11-11
6,2015-01-07,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,20.4,25.4,22.3,51,5.87,Open-Meteo Historical Weather,2026-11-11
7,2015-01-08,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,19.5,23.2,20.9,56,5.55,Open-Meteo Historical Weather,2026-11-11
8,2015-01-09,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,18.7,24.5,20.9,52,5.83,Open-Meteo Historical Weather,2026-11-11
9,2015-01-10,Dakar,Dakar,Niayes,14.6937,-17.44406,0.0,19.2,25.3,21.4,63,4.46,Open-Meteo Historical Weather,2026-11-11


In [ ]:
dataset.tail(15)

,date,localite,region,zone_agroecologique,latitude,longitude,precipitation_mm,tmin_c,tmax_c,tmean_c,rh_mean_pct,et0_mm,source,date_extraction
48201,2025-12-17,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,21.1,30.4,24.8,34,5.71,Open-Meteo Historical Weather,2026-11-11
48202,2025-12-18,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,20.0,29.2,23.6,36,5.92,Open-Meteo Historical Weather,2026-11-11
48203,2025-12-19,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,17.6,30.0,23.2,42,5.00,Open-Meteo Historical Weather,2026-11-11
48204,2025-12-20,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,19.2,32.1,24.6,41,5.22,Open-Meteo Historical Weather,2026-11-11
48205,2025-12-21,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,18.2,32.4,24.5,42,5.11,Open-Meteo Historical Weather,2026-11-11
48206,2025-12-22,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,18.4,33.3,25.2,41,5.29,Open-Meteo Historical Weather,2026-11-11
48207,2025-12-23,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,18.3,32.8,25.2,38,5.32,Open-Meteo Historical Weather,2026-11-11
48208,2025-12-24,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,20.5,33.2,25.8,37,5.38,Open-Meteo Historical Weather,2026-11-11
48209,2025-12-25,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,19.4,33.4,25.7,41,5.31,Open-Meteo Historical Weather,2026-11-11
48210,2025-12-26,Kolda,Kolda,Casamance,12.8939,-14.94125,0.0,18.4,32.6,24.6,38,5.51,Open-Meteo Historical Weather,2026-11-11


In [ ]:
dataset.shape

(48216, 14)

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48216 entries, 0 to 48215
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 48216 non-null  object 
 1   localite             48216 non-null  object 
 2   region               48216 non-null  object 
 3   zone_agroecologique  48216 non-null  object 
 4   latitude             48216 non-null  float64
 5   longitude            48216 non-null  float64
 6   precipitation_mm     48216 non-null  float64
 7   tmin_c               48216 non-null  float64
 8   tmax_c               48216 non-null  float64
 9   tmean_c              48216 non-null  float64
 10  rh_mean_pct          48216 non-null  int64  
 11  et0_mm               48216 non-null  float64
 12  source               48216 non-null  object 
 13  date_extraction      48216 non-null  object 
dtypes: float64(7), int64(1), object(6)
memory usage: 5.2+ MB


In [ ]:
dataset.columns

Index(['date', 'localite', 'region', 'zone_agroecologique', 'latitude',
       'longitude', 'precipitation_mm', 'tmin_c', 'tmax_c', 'tmean_c',
       'rh_mean_pct', 'et0_mm', 'source', 'date_extraction'],
      dtype='object')

In [ ]:
dataset['localite'].unique()

array(['Dakar', 'Mboro', 'Saint-Louis', 'Matam', 'Linguère', 'Ranérou',
       'Diourbel', 'Kaolack', 'Tambacounda', 'Kédougou', 'Ziguinchor',
       'Kolda'], dtype=object)

Les 12 localités attendues sont bien présentes : Dakar, Mboro, Saint-Louis, Matam, Linguère, Ranérou, Diourbel, Kaolack, Tambacounda, Kédougou, Ziguinchor et Kolda. Cela correspond à la sélection de l'étude, deux localités par zone.

In [ ]:
dataset['region'].unique()

array(['Dakar', 'Thiès', 'Saint-Louis', 'Matam', 'Louga', 'Diourbel',
       'Kaolack', 'Tambacounda', 'Kédougou', 'Ziguinchor', 'Kolda'],
      dtype=object)

In [ ]:
dataset['zone_agroecologique'].unique()

array(['Niayes', 'Vallée du fleuve', 'Ferlo', 'Bassin arachidier',
       'Sénégal oriental', 'Casamance'], dtype=object)

Les six zones agroécologiques de l'étude sont toutes représentées : Niayes, Vallée du fleuve, Ferlo, Bassin arachidier, Sénégal oriental et Casamance.

In [ ]:
dataset.isna().sum()

,0
date,0
localite,0
region,0
zone_agroecologique,0
latitude,0
longitude,0
precipitation_mm,0
tmin_c,0
tmax_c,0
tmean_c,0


In [ ]:
dataset.duplicated().sum()

np.int64(0)

In [ ]:
print("COUVERTURE TEMPORELLE")
print(dataset.groupby("localite").agg(
    premiere_date=("date", "min"),
    derniere_date=("date", "max"),
    nombre_jours=("date", "count")
))

COUVERTURE TEMPORELLE
            premiere_date derniere_date  nombre_jours
localite                                             
Dakar          2015-01-01    2025-12-31          4018
Diourbel       2015-01-01    2025-12-31          4018
Kaolack        2015-01-01    2025-12-31          4018
Kolda          2015-01-01    2025-12-31          4018
Kédougou       2015-01-01    2025-12-31          4018
Linguère       2015-01-01    2025-12-31          4018
Matam          2015-01-01    2025-12-31          4018
Mboro          2015-01-01    2025-12-31          4018
Ranérou        2015-01-01    2025-12-31          4018
Saint-Louis    2015-01-01    2025-12-31          4018
Tambacounda    2015-01-01    2025-12-31          4018
Ziguinchor     2015-01-01    2025-12-31          4018


Chaque localité commence au 1er janvier 2015, finit au 31 décembre 2025 et compte 4018 jours. La couverture temporelle est donc identique partout et complète, ce qui rend les localités directement comparables entre elles.

In [ ]:
print("Tmin > Tmax :",
      (dataset["tmin_c"] > dataset["tmax_c"]).sum())

Tmin > Tmax : 0


Aucun jour où la température minimale dépasse la maximale. Cette incohérence physique n'existe pas dans les données.

In [ ]:
print("humidité inférieure à 0 :",
      (dataset["rh_mean_pct"] < 0).sum())
print("humidité supérieure à 100 :",
      (dataset["rh_mean_pct"] > 100).sum())

humidité inférieure à 0 : 0
humidité supérieure à 100 : 0


L'humidité relative reste toujours entre 0 et 100 %, donc aucune valeur impossible.

In [ ]:
print("précipitations négatives :",
      (dataset["precipitation_mm"] < 0).sum())

précipitations négatives : 0


Aucune précipitation négative.

In [ ]:
print("ET0 négative :",
      (dataset["et0_mm"] < 0).sum())

ET0 négative : 0


Aucune évapotranspiration négative.

In [ ]:
# Statistiques descriptives des variables météorologiques
variables = [
    "precipitation_mm",
    "tmin_c",
    "tmax_c",
    "tmean_c",
    "rh_mean_pct",
    "et0_mm"
]

print(dataset[variables].describe().round(2))

       precipitation_mm    tmin_c    tmax_c   tmean_c  rh_mean_pct    et0_mm
count          48216.00  48216.00  48216.00  48216.00     48216.00  48216.00
mean               1.53     22.85     33.81     27.83        51.49      6.05
std                5.92      3.06      4.49      3.31        25.37      1.90
min                0.00     12.00     19.80     17.80         5.00      0.63
25%                0.00     20.60     30.60     25.90        27.00      4.65
50%                0.00     23.40     33.80     27.70        53.00      5.89
75%                0.00     25.00     37.20     29.70        76.00      7.34
max              239.40     34.00     46.90     39.80        96.00     14.95


La pluie moyenne journalière est de 1,53 mm mais la médiane et même le troisième quartile sont à 0 : plus des trois quarts des jours sont secs, ce qui est normal pour un climat à une seule saison des pluies. Le maximum de 239,4 mm en une journée est très élevé. La température minimale ne descend jamais sous 12 °C, donc le gel est impossible sur ces 12 localités. La Tmax moyenne est de 33,8 °C et le maximum de 46,9 °C. L'humidité varie de 5 à 96 %, ce qui reflète le contraste entre l'harmattan sec et la mousson. L'ET0 moyenne est de 6 mm par jour, avec un maximum de 15 mm.

In [ ]:
# Les 20 températures maximales les plus élevées
print(
    dataset.nlargest(20, "tmax_c")[
        ["date", "localite", "tmin_c", "tmax_c", "tmean_c"]
    ]
)

             date     localite  tmin_c  tmax_c  tmean_c
12568  2016-05-29        Matam    32.5    46.9     39.6
15499  2024-06-07        Matam    32.4    46.6     39.7
15137  2023-06-11        Matam    32.2    46.2     39.5
15474  2024-05-13        Matam    30.7    46.2     39.1
20604  2016-05-29      Ranérou    27.9    46.1     37.6
15475  2024-05-14        Matam    32.6    46.0     39.6
15486  2024-05-25        Matam    32.9    46.0     39.4
15487  2024-05-26        Matam    33.6    46.0     39.8
15434  2024-04-03        Matam    32.2    45.9     39.3
15504  2024-06-12        Matam    30.0    45.9     37.9
35523  2024-04-02  Tambacounda    28.8    45.9     36.9
15472  2024-05-11        Matam    31.2    45.8     38.7
15483  2024-05-22        Matam    33.0    45.8     39.5
14744  2022-05-14        Matam    27.4    45.7     36.5
15433  2024-04-02        Matam    31.1    45.7     38.7
23470  2024-04-03      Ranérou    31.4    45.7     38.6
23508  2024-05-11      Ranérou    27.7    45.7  

Les températures les plus élevées sont presque toutes à Matam (et quelques-unes à Ranérou et Tambacounda), entre avril et juin, juste avant l'arrivée de la saison des pluies. C'est cohérent avec le climat sahélien de la vallée du fleuve. L'année 2024 revient souvent, ce qui suggère un printemps particulièrement chaud cette année-là. Les Tmin élevées (30 à 33 °C) montrent aussi que les nuits restent très chaudes pendant ces épisodes.

In [ ]:
print("Journées avec Tmax > 40 °C :")
print(
    dataset[dataset["tmax_c"] > 40]
    .groupby("localite")
    .size()
    .sort_values(ascending=False)
)

print("\nJournées avec Tmax > 45 °C :")
print(
    dataset[dataset["tmax_c"] > 45]
    .groupby("localite")
    .size()
    .sort_values(ascending=False)
)

Journées avec Tmax > 40 °C :
localite
Matam          860
Tambacounda    759
Ranérou        653
Kédougou       448
Linguère       445
Kolda          308
Kaolack        245
Diourbel       208
Ziguinchor      97
Saint-Louis     37
Mboro            1
dtype: int64

Journées avec Tmax > 45 °C :
localite
Matam          34
Ranérou        14
Linguère        4
Tambacounda     3
Kaolack         1
dtype: int64


Le nombre de jours au-dessus de 40 °C suit un gradient net : Matam (860), Tambacounda (759) et Ranérou (653) sont les plus touchées, tandis que les localités côtières comme Mboro (1 jour) et Saint-Louis (37 jours) sont bien plus modérées grâce à l'influence maritime décrite dans l'étude. Au-dessus de 45 °C, seules Matam (34 jours), Ranérou, Linguère, Tambacounda et Kaolack sont concernées. Ces chiffres seront utiles pour fixer les seuils de chaleur des profils de culture.

In [ ]:
dataset["date"] = pd.to_datetime(dataset["date"])

extremes = dataset.nlargest(3, "tmax_c")

for _, ligne in extremes.iterrows():

    date_extreme = ligne["date"]
    localite = ligne["localite"]

    print(f"\n{localite} - {date_extreme.date()}")

    periode = dataset[
        (dataset["localite"] == localite) &
        (dataset["date"].between(
            date_extreme - pd.Timedelta(days=3),
            date_extreme + pd.Timedelta(days=3)
        ))
    ]

    print(
        periode[
            ["date", "tmin_c", "tmax_c", "tmean_c"]
        ].to_string(index=False)
    )


Matam - 2016-05-29
      date  tmin_c  tmax_c  tmean_c
2016-05-26    25.3    39.3     33.0
2016-05-27    27.9    40.2     34.1
2016-05-28    28.5    41.9     35.8
2016-05-29    32.5    46.9     39.6
2016-05-30    31.6    44.3     38.5
2016-05-31    28.8    42.1     36.0
2016-06-01    28.8    41.4     35.8

Matam - 2024-06-07
      date  tmin_c  tmax_c  tmean_c
2024-06-04    27.2    41.5     35.0
2024-06-05    28.7    43.1     36.5
2024-06-06    30.6    45.0     38.0
2024-06-07    32.4    46.6     39.7
2024-06-08    30.0    40.7     36.1
2024-06-09    29.7    39.1     34.2
2024-06-10    29.1    40.3     35.2

Matam - 2023-06-11
      date  tmin_c  tmax_c  tmean_c
2023-06-08    31.9    44.3     38.2
2023-06-09    32.1    44.0     38.0
2023-06-10    32.3    44.3     38.7
2023-06-11    32.2    46.2     39.5
2023-06-12    30.2    42.9     37.5
2023-06-13    30.7    41.6     36.3
2023-06-14    29.0    39.7     34.7


On regarde la semaine autour des trois valeurs les plus extrêmes. Dans chaque cas, la température maximale monte progressivement pendant plusieurs jours (41, 42, puis 46 °C) avant de redescendre, et la Tmin et la Tmean suivent. Ce sont donc de vraies vagues de chaleur et non des erreurs isolées du capteur ou de l'API. Ces valeurs extrêmes sont conservées dans le jeu de données, elles sont physiquement plausibles pour Matam en mai-juin.

In [ ]:
import numpy as np
dataset["date"] = pd.to_datetime(dataset["date"])

# Les 10 plus fortes précipitations journalières
print(dataset.nlargest(10, "precipitation_mm")[["date", "localite", "precipitation_mm"]].to_string(index=False))

      date    localite  precipitation_mm
2019-08-22    Diourbel             239.4
2022-08-18       Matam             172.2
2020-09-06     Kaolack             172.0
2024-09-06    Kédougou             169.5
2018-08-30  Ziguinchor             167.5
2020-09-02     Ranérou             149.9
2019-08-18       Kolda             143.6
2020-08-29    Kédougou             139.1
2020-09-05 Tambacounda             137.7
2021-08-19       Kolda             136.0


Les plus fortes pluies sont à regarder avec prudence. Une valeur de plus de 200 mm en un jour est possible lors d'un épisode exceptionnel, mais les données viennent d'une réanalyse (ERA5) et non de pluviomètres, donc ces extrêmes peuvent être surestimés ou lissés. On ne supprime pas ces valeurs car rien ne prouve qu'elles soient fausses, mais on les garde en tête : elles pèsent beaucoup sur les cumuls de pluie sur 3, 7 ou 14 jours.

In [ ]:
# Cumul de pluie annuel par localité (moyenne, minimum et maximum sur 2015-2025)
dataset["annee"] = dataset["date"].dt.year
cumul_annuel = (dataset.groupby(["zone_agroecologique", "localite", "annee"])["precipitation_mm"]
                .sum().reset_index())
resume_pluie = (cumul_annuel.groupby(["zone_agroecologique", "localite"])["precipitation_mm"]
                .agg(moyenne="mean", minimum="min", maximum="max").round(0))
print(resume_pluie)

                                 moyenne  minimum  maximum
zone_agroecologique localite                              
Bassin arachidier   Diourbel       460.0    323.0    663.0
                    Kaolack        622.0    433.0    942.0
Casamance           Kolda         1049.0    776.0   1364.0
                    Ziguinchor    1213.0    906.0   1620.0
Ferlo               Linguère       320.0    165.0    512.0
                    Ranérou        346.0    168.0    614.0
Niayes              Dakar          324.0    176.0    471.0
                    Mboro          287.0    180.0    441.0
Sénégal oriental    Kédougou       952.0    628.0   1480.0
                    Tambacounda    702.0    427.0    988.0
Vallée du fleuve    Matam          292.0    140.0    675.0
                    Saint-Louis    160.0     69.0    264.0


Ce tableau sert à vérifier que le gradient de pluie nord-sud décrit dans l'étude se retrouve dans les données : de l'ordre de 250 à 500 mm pour le Ferlo et la vallée, 400 à 700 mm pour le bassin arachidier, 600 à 1100 mm pour le Sénégal oriental et 900 à 1200 mm pour la Casamance. Si une localité s'écarte beaucoup de la fourchette de l'étude, c'est normal à petite échelle (la réanalyse a une maille de plusieurs dizaines de kilomètres) La grande différence entre le minimum et le maximum d'une même localité montre aussi la forte variabilité d'une année à l'autre.

In [ ]:
# Proportion de jours secs (aucune pluie) et de jours pluvieux (>= 1 mm) par zone
SEUIL_JOUR_PLUVIEUX_MM = 1.0
jours = dataset.assign(
    jour_sec=(dataset["precipitation_mm"] == 0),
    jour_pluvieux=(dataset["precipitation_mm"] >= SEUIL_JOUR_PLUVIEUX_MM),
)
print((jours.groupby("zone_agroecologique")[["jour_sec", "jour_pluvieux"]].mean() * 100).round(1))

                     jour_sec  jour_pluvieux
zone_agroecologique                         
Bassin arachidier        76.5           15.2
Casamance                65.3           27.2
Ferlo                    82.0           10.6
Niayes                   82.4           10.5
Sénégal oriental         66.6           23.6
Vallée du fleuve         85.6            7.9


Cette proportion doit refléter le caractère très saisonnier du climat : la grande majorité des jours sont secs dans toutes les zones, avec une part de jours pluvieux qui augmente vers le sud et l'est. Le seuil de 1 mm pour définir un jour pluvieux est une convention courante en climatologie, mais c'est un choix de paramètre. Il est gardé dans une variable pour pouvoir le changer facilement.

In [ ]:
# Vérification de la continuité du calendrier : jours manquants et doublons sur (date, localite)
calendrier = pd.date_range("2015-01-01", "2025-12-31", freq="D")

controle_dates = []
for loc, g in dataset.groupby("localite"):
    controle_dates.append({
        "localite": loc,
        "dates_manquantes": len(calendrier.difference(g["date"])),
        "doublons_date": int(g["date"].duplicated().sum()),
    })
print(pd.DataFrame(controle_dates).to_string(index=False))

   localite  dates_manquantes  doublons_date
      Dakar                 0              0
   Diourbel                 0              0
    Kaolack                 0              0
      Kolda                 0              0
   Kédougou                 0              0
   Linguère                 0              0
      Matam                 0              0
      Mboro                 0              0
    Ranérou                 0              0
Saint-Louis                 0              0
Tambacounda                 0              0
 Ziguinchor                 0              0


In [ ]:
VARIABLES = ["precipitation_mm", "tmin_c", "tmax_c", "rh_mean_pct", "et0_mm"]

def controler_qualite(df):
    """Rapport de qualité par localité : complétude, anomalies et doublons."""
    lignes = []
    for loc, g in df.groupby("localite"):
        ligne = {"localite": loc, "nb_lignes": len(g)}
        for v in VARIABLES:
            ligne[f"completude_{v}_pct"] = round(100 * g[v].notna().mean(), 2)
        ligne["tmin_sup_tmax"] = int((g["tmin_c"] > g["tmax_c"]).sum())
        ligne["pluie_negative"] = int((g["precipitation_mm"] < 0).sum())
        ligne["humidite_hors_0_100"] = int(((g["rh_mean_pct"] < 0) | (g["rh_mean_pct"] > 100)).sum())
        ligne["et0_negative"] = int((g["et0_mm"] < 0).sum())
        ligne["doublons_date"] = int(g["date"].duplicated().sum())
        lignes.append(ligne)
    return pd.DataFrame(lignes)

rapport_qualite = controler_qualite(dataset)
rapport_qualite.to_csv(f"{DOSSIER_DONNEES}/rapport_qualite_brut.csv", index=False, encoding="utf-8-sig")
print(rapport_qualite.T)

                                    0         1        2      3         4   \
localite                         Dakar  Diourbel  Kaolack  Kolda  Kédougou   
nb_lignes                         4018      4018     4018   4018      4018   
completude_precipitation_mm_pct  100.0     100.0    100.0  100.0     100.0   
completude_tmin_c_pct            100.0     100.0    100.0  100.0     100.0   
completude_tmax_c_pct            100.0     100.0    100.0  100.0     100.0   
completude_rh_mean_pct_pct       100.0     100.0    100.0  100.0     100.0   
completude_et0_mm_pct            100.0     100.0    100.0  100.0     100.0   
tmin_sup_tmax                        0         0        0      0         0   
pluie_negative                       0         0        0      0         0   
humidite_hors_0_100                  0         0        0      0         0   
et0_negative                         0         0        0      0         0   
doublons_date                        0         0        0      0

Ce rapport résume la qualité par localité sous forme de fonction réutilisable. Les complétudes à 100 % et les compteurs d'anomalies à zéro confirment les contrôles faits plus haut. Le même code pourra être relancé sur un autre fournisseur (CSV, NASA POWER) sans rien changer. La complétude sert plus tard au moteur de décision : sous un seuil configuré, le résultat devient indéterminé.

In [ ]:
# Nettoyage et construction du jeu de données propre
propre = dataset.drop(columns=["annee"]).copy()

# 1. Suppression des doublons sur (date, localite)
propre = propre.drop_duplicates(subset=["date", "localite"], keep="first")

# 2. Valeurs impossibles -> NaN (jamais remplacées par 0, surtout pour la pluie)
propre.loc[propre["precipitation_mm"] < 0, "precipitation_mm"] = np.nan
propre.loc[propre["et0_mm"] < 0, "et0_mm"] = np.nan
hors_plage = (propre["rh_mean_pct"] < 0) | (propre["rh_mean_pct"] > 100)
propre.loc[hors_plage, "rh_mean_pct"] = np.nan
incoherent = propre["tmin_c"] > propre["tmax_c"]
propre.loc[incoherent, ["tmin_c", "tmax_c", "tmean_c"]] = np.nan

# 3. Calendrier complet : une ligne par jour et par localité (les trous deviennent des NaN visibles)
grille = pd.MultiIndex.from_product(
    [sorted(propre["localite"].unique()), calendrier], names=["localite", "date"]
).to_frame(index=False)

meta = propre[["localite", "region", "zone_agroecologique", "latitude", "longitude",
               "source", "date_extraction"]].drop_duplicates("localite")

propre = (grille.merge(meta, on="localite", how="left")
                .merge(propre[["localite", "date", "precipitation_mm", "tmin_c", "tmax_c",
                               "tmean_c", "rh_mean_pct", "et0_mm"]],
                       on=["localite", "date"], how="left"))

colonnes_finales = ["date", "localite", "region", "zone_agroecologique", "latitude", "longitude",
                    "precipitation_mm", "tmin_c", "tmax_c", "tmean_c", "rh_mean_pct", "et0_mm",
                    "source", "date_extraction"]
propre = propre[colonnes_finales].sort_values(["localite", "date"]).reset_index(drop=True)

propre.to_csv(f"{DOSSIER_DONNEES}/agri_meteo_propre_2015_2025.csv", index=False, encoding="utf-8-sig")
print("Dimensions :", propre.shape)
print("Valeurs manquantes restantes :")
print(propre[VARIABLES].isna().sum())

Dimensions : (48216, 14)
Valeurs manquantes restantes :
precipitation_mm    0
tmin_c              0
tmax_c              0
rh_mean_pct         0
et0_mm              0
dtype: int64


In [ ]:
# Fixture : petit jeu de données fixe pour les tests et la démonstration hors ligne
fixture = propre[(propre["localite"] == "Kaolack") &
                 (propre["date"].between("2025-07-01", "2025-09-30"))]
fixture.to_csv(f"{DOSSIER_DONNEES}/fixture_kaolack_2025.csv", index=False, encoding="utf-8-sig")
print("Fixture :", fixture.shape, "lignes (Kaolack, 1er juillet au 30 septembre 2025)")

Fixture : (92, 14) lignes (Kaolack, 1er juillet au 30 septembre 2025)


La fixture est un extrait de trois mois pour Kaolack pendant la saison des pluies.

On transforme maintenant les données journalières en indicateurs utiles pour l'agriculture. Les indicateurs sont calculés pour chaque jour de référence, sur les 3, 7, 14 et 30 derniers jours, de façon à comparer le très court terme (quelques jours) et le mois écoulé.

In [ ]:
FENETRES = [3, 7, 14, 30]
SEUIL_JOUR_PLUVIEUX_MM = 1.0   # un jour est pluvieux à partir de 1 mm (paramètre à documenter)
SEUIL_GEL_C = 0.0              # un jour de gel est un jour où Tmin <= 0 °C (paramètre)

def plus_longue_serie_seche(pluies):
    """Plus longue suite de jours consécutifs sans pluie significative (< seuil)."""
    maximum, courante = 0, 0
    for p in pluies:
        if p < SEUIL_JOUR_PLUVIEUX_MM:
            courante += 1
            maximum = max(maximum, courante)
        else:
            courante = 0
    return maximum

def calculer_indicateurs(df):
    """Calcule les indicateurs glissants (3, 7, 14, 30 jours) pour chaque localité."""
    df = df.sort_values(["localite", "date"]).copy()
    df["jour_pluvieux"] = (df["precipitation_mm"] >= SEUIL_JOUR_PLUVIEUX_MM).astype(float)
    df.loc[df["precipitation_mm"].isna(), "jour_pluvieux"] = np.nan
    df["jour_gel"] = (df["tmin_c"] <= SEUIL_GEL_C).astype(float)
    df.loc[df["tmin_c"].isna(), "jour_gel"] = np.nan
    df["amplitude"] = df["tmax_c"] - df["tmin_c"]

    blocs = []
    for loc, g in df.groupby("localite", sort=False):
        g = g.set_index("date")
        res = pd.DataFrame(index=g.index)
        for n in FENETRES:
            roul = lambda col: g[col].rolling(n, min_periods=n)   # fenêtre incomplète -> NaN
            res[f"rain_sum_{n}d"] = roul("precipitation_mm").sum()
            res[f"wet_days_{n}d"] = roul("jour_pluvieux").sum()
            res[f"max_dry_spell_{n}d"] = roul("precipitation_mm").apply(plus_longue_serie_seche, raw=True)
            res[f"tmean_{n}d"] = roul("tmean_c").mean()
            res[f"tmin_min_{n}d"] = roul("tmin_c").min()
            res[f"tmax_max_{n}d"] = roul("tmax_c").max()
            res[f"frost_days_{n}d"] = roul("jour_gel").sum()
            res[f"diurnal_range_{n}d"] = roul("amplitude").mean()
            res[f"et0_sum_{n}d"] = roul("et0_mm").sum()
            res[f"water_balance_{n}d"] = res[f"rain_sum_{n}d"] - res[f"et0_sum_{n}d"]
            res[f"rh_mean_{n}d"] = roul("rh_mean_pct").mean()
        res.insert(0, "localite", loc)
        blocs.append(res.reset_index())
    return pd.concat(blocs, ignore_index=True)

indicateurs = calculer_indicateurs(propre)
indicateurs.to_csv(f"{DOSSIER_DONNEES}/agri_meteo_indicateurs_2015_2025.csv", index=False, encoding="utf-8-sig")
print(indicateurs.shape)

(48216, 46)


Chaque ligne du tableau d'indicateurs correspond à un jour de référence et à une localité, et résume la période qui se termine ce jour-là. Pour 4 fenêtres et 11 indicateurs, on obtient 44 colonnes en plus de la date et de la localité. Les 2, 6, 13 et 29 premiers jours de la série n'ont pas assez d'historique pour la fenêtre complète : ils sont vides (NaN), c'est voulu. Le water_balance est une différence simple entre pluie et ET0, ce n'est pas un bilan hydrique du sol. Le frost_days vaudra zéro partout ici, puisque la Tmin ne descend jamais sous 12 °C.

In [ ]:
# Exemple lisible : indicateurs de Kaolack au 15 septembre 2025, pour chaque fenêtre
ligne = indicateurs[(indicateurs["localite"] == "Kaolack") &
                    (indicateurs["date"] == "2025-09-15")].iloc[0]

noms = ["rain_sum", "wet_days", "max_dry_spell", "tmean", "tmin_min", "tmax_max",
        "frost_days", "diurnal_range", "et0_sum", "water_balance", "rh_mean"]
exemple = pd.DataFrame({n: [round(ligne[f"{n}_{f}d"], 1) for f in FENETRES] for n in noms},
                       index=[f"{f} jours" for f in FENETRES]).T
print(exemple)

               3 jours  7 jours  14 jours  30 jours
rain_sum          29.2     71.5     137.5     288.1
wet_days           3.0      7.0      12.0      24.0
max_dry_spell      0.0      0.0       2.0       2.0
tmean             27.2     26.7      26.9      26.8
tmin_min          23.1     23.1      23.1      22.7
tmax_max          32.5     32.5      33.6      33.6
frost_days         0.0      0.0       0.0       0.0
diurnal_range      8.3      7.6       7.5       7.6
et0_sum           13.5     28.5      56.2     118.9
water_balance     15.7     43.0      81.3     169.2
rh_mean           85.0     86.7      85.6      85.3


On lit ce tableau en comparant les colonnes : si la pluie des 3 derniers jours est faible alors que celle des 30 derniers jours est élevée, la période est humide mais vient de s'assécher, ce qui n'aurait pas été visible avec une seule fenêtre. C'est la raison d'utiliser plusieurs fenêtres. En septembre, au cœur de la saison des pluies, on s'attend à un bilan pluie moins ET0 plus favorable qu'en saison sèche, et à une humidité plus élevée.

In [ ]:
# Comparaison saisonnière : bilan pluie - ET0 sur 30 jours, moyenne par zone et par mois
indic = indicateurs.merge(propre[["localite", "date", "zone_agroecologique"]], on=["localite", "date"])
indic["mois"] = indic["date"].dt.month
bilan_mensuel = (indic.groupby(["zone_agroecologique", "mois"])["water_balance_30d"]
                 .mean().unstack().round(0))
print(bilan_mensuel)

mois                    1      2      3      4      5      6      7      8   \
zone_agroecologique                                                           
Bassin arachidier   -190.0 -213.0 -232.0 -241.0 -233.0 -195.0 -118.0   10.0   
Casamance           -179.0 -201.0 -216.0 -216.0 -207.0 -157.0   19.0  229.0   
Ferlo               -199.0 -222.0 -245.0 -270.0 -275.0 -243.0 -165.0  -70.0   
Niayes              -167.0 -175.0 -170.0 -159.0 -154.0 -144.0 -127.0  -53.0   
Sénégal oriental    -196.0 -215.0 -230.0 -241.0 -235.0 -178.0  -38.0   98.0   
Vallée du fleuve    -215.0 -232.0 -243.0 -251.0 -250.0 -226.0 -173.0 -101.0   

mois                    9      10     11     12  
zone_agroecologique                              
Bassin arachidier    108.0  -36.0 -144.0 -177.0  
Casamance            267.0   79.0  -84.0 -153.0  
Ferlo                 -7.0 -106.0 -181.0 -195.0  
Niayes                15.0  -75.0 -144.0 -159.0  
Sénégal oriental     166.0   19.0 -119.0 -176.0  
Vallée du fleuve 

Ce tableau permet de voir en même temps le gradient nord-sud et la saison. On s'attend à un bilan négatif presque toute l'année dans le nord (Ferlo, vallée) puisque l'ET0 y dépasse la pluie, et à un bilan positif seulement pendant les mois les plus pluvieux (surtout août et septembre), surtout dans le sud et l'est. Cela montre l'intérêt du bilan pluie moins ET0 : il indique à quel moment la pluie couvre vraiment la demande en eau de l'atmosphère, ce qui est plus informatif que la pluie seule.

In [ ]:
import yaml, os, copy

PROFILS = {
    "tomate": {
        "profile_id": "tomate", "label": "Tomate (maraîchage, Niayes et vallée)",
        "version": "0.1.0",
        "criteres": [
            {"id": "chaleur", "indicator": "tmax_max", "fenetre_jours": 30, "sens": "max",
             "vigilance": 35.0, "unfavorable": 40.0, "unite": "°C", "blocking": True,
             "source": "FAO EcoCrop et littérature sur la nouaison"},
            {"id": "froid", "indicator": "tmin_min", "fenetre_jours": 30, "sens": "min",
             "vigilance": 12.0, "unfavorable": 10.0, "unite": "°C", "blocking": True,
             "source": "FAO EcoCrop"},
            {"id": "gel", "indicator": "frost_days", "fenetre_jours": 30, "sens": "max",
             "vigilance": None, "unfavorable": 1.0, "unite": "jours", "blocking": True,
             "source": "Sensibilité de la tomate au gel"},
            {"id": "bilan_pluie_et0", "indicator": "water_balance", "fenetre_jours": 30, "sens": "min",
             "vigilance": -150.0, "unfavorable": None, "unite": "mm", "blocking": False,
             "source": "FAO 56, logique ET0 comme demande en eau ; culture irriguée"},
            {"id": "humidite", "indicator": "rh_mean", "fenetre_jours": 30, "sens": "max",
             "vigilance": 80.0, "unfavorable": None, "unite": "%", "blocking": False,
             "source": "Risque de maladies foliaires en air humide"},
        ],
    },
    "riz": {
        "profile_id": "riz", "label": "Riz (vallée du fleuve et Casamance)",
        "version": "0.1.0",
        "criteres": [
            {"id": "chaleur", "indicator": "tmax_max", "fenetre_jours": 30, "sens": "max",
             "vigilance": 35.0, "unfavorable": 38.0, "unite": "°C", "blocking": True,
             "source": "IRRI, stérilité des épillets à la floraison"},
            {"id": "froid", "indicator": "tmin_min", "fenetre_jours": 30, "sens": "min",
             "vigilance": 18.0, "unfavorable": 15.0, "unite": "°C", "blocking": True,
             "source": "IRRI, sensibilité au froid"},
            {"id": "pluie_cumulee", "indicator": "rain_sum", "fenetre_jours": 30, "sens": "min",
             "vigilance": 150.0, "unfavorable": None, "unite": "mm", "blocking": False,
             "source": "Besoins en eau du riz pluvial, FAO"},
            {"id": "gel", "indicator": "frost_days", "fenetre_jours": 30, "sens": "max",
             "vigilance": None, "unfavorable": 1.0, "unite": "jours", "blocking": True,
             "source": "Sensibilité du riz au froid extrême"},
        ],
    },
    "arachide": {
        "profile_id": "arachide", "label": "Arachide (bassin arachidier)",
        "version": "0.1.0",
        "criteres": [
            {"id": "chaleur", "indicator": "tmax_max", "fenetre_jours": 30, "sens": "max",
             "vigilance": 38.0, "unfavorable": 42.0, "unite": "°C", "blocking": True,
             "source": "FAO EcoCrop"},
            {"id": "secheresse", "indicator": "max_dry_spell", "fenetre_jours": 30, "sens": "max",
             "vigilance": 10.0, "unfavorable": 20.0, "unite": "jours", "blocking": True,
             "source": "Sensibilité à la sécheresse en floraison et formation des gousses"},
            {"id": "pluie_cumulee", "indicator": "rain_sum", "fenetre_jours": 30, "sens": "min",
             "vigilance": 60.0, "unfavorable": None, "unite": "mm", "blocking": False,
             "source": "Besoins en eau de l'arachide pluviale, FAO"},
        ],
    },
    "mil": {
        "profile_id": "mil", "label": "Mil (bassin arachidier, Ferlo, Sénégal oriental)",
        "version": "0.1.0",
        "criteres": [
            {"id": "chaleur", "indicator": "tmax_max", "fenetre_jours": 30, "sens": "max",
             "vigilance": 40.0, "unfavorable": 45.0, "unite": "°C", "blocking": True,
             "source": "FAO EcoCrop, culture tolérante à la chaleur"},
            {"id": "secheresse", "indicator": "max_dry_spell", "fenetre_jours": 30, "sens": "max",
             "vigilance": 15.0, "unfavorable": 25.0, "unite": "jours", "blocking": True,
             "source": "Tolérance du mil à la sécheresse"},
            {"id": "pluie_cumulee", "indicator": "rain_sum", "fenetre_jours": 30, "sens": "min",
             "vigilance": 50.0, "unfavorable": None, "unite": "mm", "blocking": False,
             "source": "Besoins en eau du mil pluvial"},
        ],
    },
}

DOSSIER_REGLES = f"{DOSSIER_DONNEES}/regles"
os.makedirs(DOSSIER_REGLES, exist_ok=True)
for cid, profil in PROFILS.items():
    with open(f"{DOSSIER_REGLES}/{cid}.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(profil, f, allow_unicode=True, sort_keys=False)
print("Profils écrits :", list(PROFILS))

Profils écrits : ['tomate', 'riz', 'arachide', 'mil']


Chaque profil est un fichier YAML versionné, avec une liste de critères. Un critère précise l'indicateur utilisé, la fenêtre, le sens (une valeur trop haute ou trop basse est mauvaise), le seuil de vigilance, le seuil défavorable, s'il est bloquant et sa source. Les critères bloquants sont ceux qui, s'ils sont franchis, suffisent à empêcher une bonne favorabilité (chaleur extrême, froid, gel, longue sécheresse). Les critères non bloquants (humidité, bilan d'eau, pluie cumulée) peuvent seulement baisser le niveau à favorable avec réserves, car pour une culture irriguée la pluie seule ne décide pas. Le profil de la tomate et du riz couvre les cultures irriguées de la vallée, celui de l'arachide et du mil les cultures pluviales du bassin arachidier, ce qui reprend les zones de l'étude.

In [ ]:
# Contrôle automatique des profils (critère P-06) : chaque seuil a une source
def verifier_profil(profil):
    problemes = []
    for c in profil["criteres"]:
        a_un_seuil = c["vigilance"] is not None or c["unfavorable"] is not None
        if a_un_seuil and not c.get("source"):
            problemes.append(f"critère {c['id']} : seuil sans source")
        if (c["vigilance"] is not None and c["unfavorable"] is not None):
            if c["sens"] == "max" and c["vigilance"] > c["unfavorable"]:
                problemes.append(f"critère {c['id']} : vigilance au-dessus du seuil défavorable")
            if c["sens"] == "min" and c["vigilance"] < c["unfavorable"]:
                problemes.append(f"critère {c['id']} : vigilance en dessous du seuil défavorable")
    return problemes

for cid, profil in PROFILS.items():
    print(cid, "->", verifier_profil(profil) or "OK")

tomate -> OK
riz -> OK
arachide -> OK
mil -> OK


Ce contrôle empêche deux erreurs fréquentes : un seuil sans source, et des seuils dans le mauvais ordre (par exemple une vigilance plus sévère que le seuil défavorable).

Moteur d'évaluation. Le résultat est indéterminé si la complétude des données est trop faible ou si trop peu de critères sont évaluables. Il est de faible favorabilité si au moins un critère bloquant est défavorable. Il est favorable avec réserves si aucun critère bloquant n'est défavorable mais qu'au moins un critère demande de la vigilance, n'est pas évaluable ou est défavorable sans être bloquant. Il est favorable seulement si tous les critères sont satisfaits. Le moteur est une fonction pure : mêmes entrées, mêmes sorties, et il écrit une explication en français pour chaque critère.

In [ ]:
NIVEAUX = {
    "favorable": "Favorable",
    "favorable_sous_reserve": "Favorable avec réserves",
    "peu_favorable": "Faible favorabilité",
    "indetermine": "Indéterminé",
}

SEUIL_COMPLETUDE_PCT = 90.0     # sous ce seuil -> indéterminé
MIN_CRITERES_EVALUABLES = 2     # moins de critères évaluables -> indéterminé

def evaluer_critere(critere, valeur):
    """Compare un indicateur aux seuils d'un critère. Retourne (statut, explication)."""
    ind, fen, unite = critere["indicator"], critere["fenetre_jours"], critere["unite"]
    vig, unf, sens = critere["vigilance"], critere["unfavorable"], critere["sens"]

    if valeur is None or pd.isna(valeur):
        return "non_evaluable", f"{ind} sur {fen} jours : donnée manquante, critère non évaluable."
    if vig is None and unf is None:
        return "non_evaluable", f"{ind} sur {fen} jours : seuil non défini pour ce profil."

    franchi = (lambda v, s: v >= s) if sens == "max" else (lambda v, s: v <= s)
    mot = "atteint ou dépasse" if sens == "max" else "est égal ou inférieur à"
    texte = f"{ind} sur {fen} jours = {valeur:.1f} {unite}"

    if unf is not None and franchi(valeur, unf):
        return "defavorable", f"{texte}, ce qui {mot} le seuil défavorable de {unf:g} {unite}."
    if vig is not None and franchi(valeur, vig):
        return "vigilance", f"{texte}, ce qui {mot} le seuil de vigilance de {vig:g} {unite}."
    return "ok", f"{texte}, dans la limite acceptable pour ce profil."

def evaluer(profil, indicateurs_jour, completude_pct,
            seuil_completude=SEUIL_COMPLETUDE_PCT, min_criteres=MIN_CRITERES_EVALUABLES):
    """Évalue la favorabilité météo d'une culture. indicateurs_jour : dict ou Series."""
    criteres = []
    for c in profil["criteres"]:
        valeur = indicateurs_jour.get(f"{c['indicator']}_{c['fenetre_jours']}d", np.nan)
        statut, explication = evaluer_critere(c, valeur)
        criteres.append({"id": c["id"], "status": statut, "blocking": c["blocking"],
                         "explanation": explication, "source": c["source"]})

    statuts = [c["status"] for c in criteres]
    nb_evaluables = sum(s != "non_evaluable" for s in statuts)
    limites = ["Résolution de la réanalyse de l'ordre de plusieurs dizaines de kilomètres, pas de la parcelle.",
               "Favorabilité météorologique seulement : ni humidité du sol mesurée, ni cultivabilité agronomique."]

    if completude_pct < seuil_completude:
        niveau = "indetermine"
        resume = f"Données insuffisantes : complétude de {completude_pct:.1f} % pour un minimum de {seuil_completude:g} %."
    elif nb_evaluables < min_criteres:
        niveau = "indetermine"
        resume = f"Seulement {nb_evaluables} critère(s) évaluable(s) sur {len(criteres)}, minimum requis : {min_criteres}."
    elif any(c["status"] == "defavorable" and c["blocking"] for c in criteres):
        bloquants = [c["id"] for c in criteres if c["status"] == "defavorable" and c["blocking"]]
        niveau = "peu_favorable"
        resume = "Au moins un critère bloquant est défavorable : " + ", ".join(bloquants) + "."
    elif any(s in ("vigilance", "non_evaluable", "defavorable") for s in statuts):
        niveau = "favorable_sous_reserve"
        resume = "Aucun critère bloquant défavorable, mais au moins un critère demande de la vigilance."
    else:
        niveau = "favorable"
        resume = "Tous les critères évalués sont satisfaits."

    if completude_pct < seuil_completude or nb_evaluables < len(criteres) / 2:
        confiance = "faible"
    elif completude_pct < 98:
        confiance = "moyenne"
    else:
        confiance = "elevee"

    return {"crop": profil["profile_id"], "level": niveau, "level_label": NIVEAUX[niveau],
            "summary": resume, "criteria": criteres, "completeness_pct": round(completude_pct, 1),
            "confidence": confiance, "limits": limites}

Quelques choix de conception à retenir. Un seuil franchi veut dire atteint (supérieur ou égal pour un maximum, inférieur ou égal pour un minimum), ce qui est important pour les tests aux bornes. Un critère bloquant défavorable l'emporte sur tout le reste, ce qui garantit qu'un gel ou une chaleur extrême n'est jamais compensé par une forte pluie. Les seuils des profils sont des valeurs de travail issues de références agronomiques générales (FAO, IRRI), citées dans le champ source de chaque critère. L'indice de confiance n'est pas une probabilité : il résume la qualité des données (complétude et nombre de critères évaluables).

In [ ]:
def completude_fenetre(df, localite, date_ref, jours=30):
    """Pourcentage de valeurs présentes (5 variables) sur les `jours` derniers jours."""
    debut = pd.Timestamp(date_ref) - pd.Timedelta(days=jours - 1)
    g = df[(df["localite"] == localite) & (df["date"].between(debut, pd.Timestamp(date_ref)))]
    if g.empty:
        return 0.0
    return 100 * g[VARIABLES].notna().to_numpy().mean()

indic_idx = indicateurs.set_index(["localite", "date"])

def analyser(localite, date_ref, crop_id, df=None, indic=None, profils=None):
    """Analyse complète : indicateurs + complétude + règles de la culture."""
    df = propre if df is None else df
    indic = indic_idx if indic is None else indic
    profils = PROFILS if profils is None else profils
    cle = (localite, pd.Timestamp(date_ref))
    ligne = indic.loc[cle] if cle in indic.index else pd.Series(dtype=float)
    comp = completude_fenetre(df, localite, date_ref)
    return evaluer(profils[crop_id], ligne, comp)

def afficher(resultat):
    print(f"{resultat['crop']} : {resultat['level_label']} (confiance {resultat['confidence']}, "
          f"complétude {resultat['completeness_pct']} %)")
    print(" ", resultat["summary"])
    for c in resultat["criteria"]:
        print(f"  - [{c['status']}] {c['explanation']}")

afficher(analyser("Kaolack", "2025-09-15", "arachide"))

arachide : Favorable (confiance elevee, complétude 100.0 %)
  Tous les critères évalués sont satisfaits.
  - [ok] tmax_max sur 30 jours = 33.6 °C, dans la limite acceptable pour ce profil.
  - [ok] max_dry_spell sur 30 jours = 2.0 jours, dans la limite acceptable pour ce profil.
  - [ok] rain_sum sur 30 jours = 288.1 mm, dans la limite acceptable pour ce profil.


Cet exemple montre le fonctionnement complet pour une localité, une date et une culture : le niveau global, la confiance et la phrase d'explication de chaque critère avec la valeur observée et le seuil franchi.

In [ ]:
# Résultats pour toutes les localités et toutes les cultures à deux dates : saison des pluies et saison sèche
dates = {"15 septembre 2025 (saison des pluies)": "2025-09-15", "15 mars 2025 (saison sèche)": "2025-03-15"}

for nom, d in dates.items():
    lignes = []
    for loc in sorted(propre["localite"].unique()):
        ligne = {"localite": loc}
        for crop in PROFILS:
            ligne[crop] = analyser(loc, d, crop)["level_label"]
        lignes.append(ligne)
    print("\n" + nom)
    print(pd.DataFrame(lignes).to_string(index=False))


15 septembre 2025 (saison des pluies)
   localite                  tomate                     riz  arachide       mil
      Dakar Favorable avec réserves Favorable avec réserves Favorable Favorable
   Diourbel Favorable avec réserves Favorable avec réserves Favorable Favorable
    Kaolack Favorable avec réserves               Favorable Favorable Favorable
      Kolda Favorable avec réserves               Favorable Favorable Favorable
   Kédougou Favorable avec réserves               Favorable Favorable Favorable
   Linguère Favorable avec réserves Favorable avec réserves Favorable Favorable
      Matam Favorable avec réserves Favorable avec réserves Favorable Favorable
      Mboro Favorable avec réserves Favorable avec réserves Favorable Favorable
    Ranérou Favorable avec réserves Favorable avec réserves Favorable Favorable
Saint-Louis Favorable avec réserves Favorable avec réserves Favorable Favorable
Tambacounda Favorable avec réserves               Favorable Favorable Favorable
 

Ce tableau compare la favorabilité des quatre cultures dans les 12 localités à deux moments contrastés. On s'attend à ce que les cultures pluviales (arachide, mil) ressortent mieux en saison des pluies qu'en saison sèche, où le manque de pluie fait monter la vigilance ou déclenche un critère bloquant de sécheresse.

Tests du moteur sur des scénarios construits

In [ ]:
def scenario(profil, **valeurs):
    """Fabrique des indicateurs 30 jours à partir de valeurs simples."""
    return {f"{k}_30d": v for k, v in valeurs.items()}

profil_test = copy.deepcopy(PROFILS["arachide"])
bon = scenario(profil_test, tmax_max=33.0, max_dry_spell=4, rain_sum=120.0)

# P-01 : mêmes données, même résultat
assert evaluer(profil_test, bon, 100) == evaluer(profil_test, bon, 100)

# P-03 : données insuffisantes -> indéterminé, jamais favorable
assert evaluer(profil_test, bon, 50)["level"] == "indetermine"
assert evaluer(profil_test, {}, 100)["level"] == "indetermine"

# Un profil dont tous les critères sont ok donne favorable
assert evaluer(profil_test, bon, 100)["level"] == "favorable"

# Une vigilance sur un critère non bloquant donne favorable avec réserves
vigilance = scenario(profil_test, tmax_max=33.0, max_dry_spell=4, rain_sum=40.0)
assert evaluer(profil_test, vigilance, 100)["level"] == "favorable_sous_reserve"

# P-04 : chaleur extrême avec forte pluie -> jamais favorable
chaud_pluvieux = scenario(profil_test, tmax_max=44.0, max_dry_spell=2, rain_sum=300.0)
assert evaluer(profil_test, chaud_pluvieux, 100)["level"] == "peu_favorable"

# P-02 : tests aux bornes (seuil atteint = franchi)
for valeur, attendu in [(37.9, "ok"), (38.0, "vigilance"), (41.9, "vigilance"), (42.0, "defavorable")]:
    statut, _ = evaluer_critere(profil_test["criteres"][0], valeur)
    assert statut == attendu, (valeur, statut)
assert evaluer_critere(profil_test["criteres"][0], np.nan)[0] == "non_evaluable"

# P-05 : au moins une explication en français pour chaque résultat
assert all(len(c["explanation"]) > 10 for c in evaluer(profil_test, chaud_pluvieux, 100)["criteria"])

# P-06 : tous les profils respectent la règle seuil -> source
assert all(verifier_profil(p) == [] for p in PROFILS.values())

print("Tous les tests du moteur passent.")

Tous les tests du moteur passent.


Chaque vérification correspond à une propriété du cahier des charges : le déterminisme, les données insuffisantes qui donnent indéterminé, un profil dont tous les critères sont satisfaits qui donne favorable, une vigilance qui donne favorable avec réserves, une forte chaleur qui n'est jamais compensée par la pluie, les bornes des seuils, la présence d'explications et la règle sur les sources. Ces tests ne mesurent pas un taux de réussite : un moteur à règles se vérifie par des scénarios, pas par une précision, puisqu'il n'existe aucune variable cible fiable pour entraîner un modèle de machine learning.

Analyse de sensibilité des seuils. Comme les seuils sont des valeurs de travail, on regarde à quel point le résultat dépend de leur valeur : on décale les seuils de chaleur du profil de la tomate de moins 2 et de plus 2 degrés, et on compare la répartition des niveaux sur de nombreuses dates et localités.

In [ ]:
def repartition_niveaux(profils, crop_id, dates_test):
    comptes = {n: 0 for n in NIVEAUX.values()}
    for loc in sorted(propre["localite"].unique()):
        for d in dates_test:
            comptes[analyser(loc, d, crop_id, profils=profils)["level_label"]] += 1
    total = sum(comptes.values())
    return {k: round(100 * v / total, 1) for k, v in comptes.items()}

# Une date tous les 15 jours sur 2023-2025 (les 29 premiers jours de la série n'ont pas de fenêtre complète)
dates_test = pd.date_range("2023-01-01", "2025-12-31", freq="15D")

sensibilite = {}
for decalage in (-2, 0, 2):
    profils_modifies = copy.deepcopy(PROFILS)
    for c in profils_modifies["tomate"]["criteres"]:
        if c["id"] == "chaleur":
            c["vigilance"] += decalage
            c["unfavorable"] += decalage
    sensibilite[f"seuils chaleur {decalage:+d} °C"] = repartition_niveaux(profils_modifies, "tomate", dates_test)

print(pd.DataFrame(sensibilite).T)

                      Favorable  Favorable avec réserves  Faible favorabilité  \
seuils chaleur -2 °C        4.8                     48.1                 47.1   
seuils chaleur +0 °C        7.7                     62.6                 29.7   
seuils chaleur +2 °C       16.1                     67.2                 16.7   

                      Indéterminé  
seuils chaleur -2 °C          0.0  
seuils chaleur +0 °C          0.0  
seuils chaleur +2 °C          0.0  


Chaque ligne donne le pourcentage de situations dans chaque niveau pour un décalage des seuils de chaleur. Plus la répartition change entre les lignes, plus le résultat dépend de la valeur exacte du seuil. Si le pourcentage de faible favorabilité varie beaucoup entre moins 2 et plus 2 degrés, c'est que beaucoup de situations sont proches du seuil. Ce résultat peut figurer dans le rapport du lot 2.